In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

# Warna tema slide
PRIMARY   = '#1F4E79'
SECONDARY = '#2E75B6'
ACCENT    = '#5BA3C9'
LIGHT     = '#BDD7EE'
TEAL      = '#7ABFB3'
GRAY      = '#666666'

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Libraries loaded!')

Libraries loaded!


In [3]:
df_clusters_raw = pd.read_csv('sml_clusters.csv')
df_ipl_raw = pd.read_csv('sml_ipl_billings.csv')
df_units_raw = pd.read_csv('sml_units.csv')

df_clusters = df_clusters_raw.copy()
df_ipl = df_ipl_raw.copy()
df_units = df_units_raw.copy()

print(f'Clusters    : {df_clusters.shape}')
print(f'IPL : {df_ipl.shape}')
print(f'Units   : {df_units.shape}')

Clusters    : (100, 3)
IPL : (300000, 9)
Units   : (25000, 6)


## **Data Cleaning**

In [4]:
# Shape dan info umum — lakukan untuk ketiga tabel
print(df_clusters.shape)
df_clusters.info()

(100, 3)
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   cluster_id        100 non-null    str  
 1   township_name     100 non-null    str  
 2   cluster_category  100 non-null    str  
dtypes: str(3)
memory usage: 6.2 KB


In [5]:
print(df_ipl.shape)
df_ipl.info()

(300000, 9)
<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   invoice_id      300000 non-null  str    
 1   unit_id         300000 non-null  str    
 2   billing_month   300000 non-null  str    
 3   water_usage_m3  300000 non-null  float64
 4   ipl_base_fee    300000 non-null  int64  
 5   total_amount    300000 non-null  float64
 6   payment_status  300000 non-null  str    
 7   payment_method  209722 non-null  str    
 8   payment_date    209722 non-null  str    
dtypes: float64(2), int64(1), str(6)
memory usage: 34.6 MB


In [6]:
print(df_units.shape)
df_units.info()

(25000, 6)
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   unit_id         25000 non-null  str    
 1   cluster_id      25000 non-null  str    
 2   owner_name      25000 non-null  str    
 3   contact_number  20000 non-null  float64
 4   is_vacant       25000 non-null  bool   
 5   handover_date   25000 non-null  str    
dtypes: bool(1), float64(1), str(4)
memory usage: 2.2 MB


In [7]:
print(f"{df_clusters.shape}")
print(f"{df_ipl.shape}")
print(f"{df_units.shape}")

(100, 3)
(300000, 9)
(25000, 6)


In [8]:
print(f"total cells = {df_clusters.shape[0] * df_clusters.shape[1]}")
print(f"total empty cells = {df_clusters.isnull().sum().sum()}")
print(f"percentage of empty cells = {(df_clusters.isnull().sum().sum())/(df_clusters.shape[0] * df_clusters.shape[1]):.2%}")

total cells = 300
total empty cells = 0
percentage of empty cells = 0.00%


In [9]:
print(f"total cells = {df_ipl.shape[0] * df_ipl.shape[1]}")
print(f"total empty cells = {df_ipl.isnull().sum().sum()}")
print(f"percentage of empty cells = {(df_ipl.isnull().sum().sum())/(df_ipl.shape[0] * df_ipl.shape[1]):.2%}")

total cells = 2700000
total empty cells = 180556
percentage of empty cells = 6.69%


In [10]:
print(f"total cells = {df_units.shape[0] * df_units.shape[1]}")
print(f"total empty cells = {df_units.isnull().sum().sum()}")
print(f"percentage of empty cells = {(df_units.isnull().sum().sum())/(df_units.shape[0] * df_units.shape[1]):.2%}")

total cells = 150000
total empty cells = 5000
percentage of empty cells = 3.33%


In [11]:
#Kolom Kritis: contact_number, handover_date, payment_method, payment_date, payment_status, water_usage_m3

#Contact Number ada yang nomor teleponnya kosong (Missing Values)
#Payment Method ada data yang duplikat (BCA Virtual Account, VA BCA, bca va)
# IPL statusnya "paid" dengan Payment Date terjadi sebelum rumah diserahkan (Handover)
#Water Usage memiliki outliers ekstrim
#Unit ID dalam 1 billing_month ada yang mempunyai lebih dari 1 invoice 

df_ipl.groupby("payment_method")["unit_id"].count()

payment_method
BCA Virtual Account    23216
Cash                   23370
Credit Card            23267
KAS                    23438
Tokopedia              23238
VA BCA                 23200
bca va                 23242
cc                     23302
toped                  23449
Name: unit_id, dtype: int64

In [12]:
df_ipl

,invoice_id,unit_id,billing_month,water_usage_m3,ipl_base_fee,total_amount,payment_status,payment_method,payment_date
0,INV-IPL-00000001,UNT-17116,2023-08,48.0,850000,1210000.0,Paid,Cash,2023-08-17
1,INV-IPL-00000002,UNT-09718,2023-10,26.0,500000,695000.0,Unpaid,NaN,NaN
2,INV-IPL-00000003,UNT-20344,2024-06,1.0,350000,357500.0,Paid,Cash,2024-06-25
3,INV-IPL-00000004,UNT-23600,2024-02,17.0,850000,977500.0,Paid,KAS,2024-02-06
4,INV-IPL-00000005,UNT-12569,2023-05,1.0,850000,857500.0,Paid,toped,2023-05-08
...,...,...,...,...,...,...,...,...,...
299995,INV-IPL-00299996,UNT-21178,2023-03,37.0,350000,627500.0,Overdue,NaN,NaN
299996,INV-IPL-00299997,UNT-01536,2024-07,42.0,500000,815000.0,Paid,Credit Card,2024-07-06
299997,INV-IPL-00299998,UNT-21834,2024-10,0.0,500000,500000.0,Paid,Cash,2024-10-22
299998,INV-IPL-00299999,UNT-03433,2024-10,11.0,850000,932500.0,Paid,cc,2024-10-27


## **1. Standarisasi Payment Method**

In [13]:
payment_method_map = {
    "Cash": "Cash",
    "KAS": "Cash",
    "Tokopedia": "Tokopedia",
    "toped": "Tokopedia",
    "BCA Virtual Account": "BCA Virtual Account",
    "bca va": "BCA Virtual Account",
    "VA BCA": "BCA Virtual Account",
    "Credit Card": "Credit Card",
    "cc": "Credit Card",
}

df_ipl["payment_method_clean"] = df_ipl["payment_method"].map(payment_method_map)

print("Before:")
print(df_ipl["payment_method"].value_counts(dropna=False))
print()
print("After:")
print(df_ipl["payment_method_clean"].value_counts(dropna=False))

Before:
payment_method
NaN                    90278
toped                  23449
KAS                    23438
Cash                   23370
cc                     23302
Credit Card            23267
bca va                 23242
Tokopedia              23238
BCA Virtual Account    23216
VA BCA                 23200
Name: count, dtype: int64

After:
payment_method_clean
NaN                    90278
BCA Virtual Account    69658
Cash                   46808
Tokopedia              46687
Credit Card            46569
Name: count, dtype: int64


## **2. Outlier di water_usage_m3**

In [14]:
#Flag dua jenis anomali
def flag_water(x):
    if x < 0:
        return "Negative (Input Error)"
    elif x == 9999:
        return "Extreme Outlier (Human Error)"
    else:
        return "Normal"


df_ipl['water_usage_flag'] = df_ipl['water_usage_m3'].apply(flag_water)
print(df_ipl['water_usage_flag'].value_counts())
print(f"\nTotal baris anomali: {(df_ipl['water_usage_flag'] != 'Normal').sum()} dari {len(df_ipl)} ({(df_ipl['water_usage_flag'] != 'Normal').mean():.2%})")

water_usage_flag
Normal                           297000
Negative (Input Error)             1519
Extreme Outlier (Human Error)      1481
Name: count, dtype: int64

Total baris anomali: 3000 dari 300000 (1.00%)


In [15]:
#Buat kolom water_usage_clean
#NaN akan diisi oleh median pemakaian air
df_ipl['water_usage_m3_clean'] = df_ipl['water_usage_m3'].where(df_ipl['water_usage_flag'] == 'Normal')

median_per_unit = df_ipl.groupby('unit_id')['water_usage_m3_clean'].transform('median')
global_median = df_ipl['water_usage_m3_clean'].median()

df_ipl['water_usage_m3_clean'] = df_ipl['water_usage_m3_clean'].fillna(median_per_unit).fillna(global_median)

print('Sebelum cleaning -> max:', df_ipl['water_usage_m3'].max(), '| min:', df_ipl['water_usage_m3'].min())
print('Sesudah cleaning -> max:', df_ipl['water_usage_m3_clean'].max(), '| min:', df_ipl['water_usage_m3_clean'].min())

Sebelum cleaning -> max: 9999.0 | min: -15.0
Sesudah cleaning -> max: 49.0 | min: 0.0


## **3. Duplikat invoice di billing_month yang sama**

In [16]:
invoice_counts = df_ipl.groupby(['unit_id', 'billing_month']).size()
print(invoice_counts)

unit_id    billing_month
UNT-00001  2020-07          1
           2023-02          1
           2023-08          4
           2023-10          2
           2023-11          1
                           ..
UNT-25000  2023-11          1
           2024-01          1
           2024-06          1
           2024-08          1
           2024-11          2
Length: 238305, dtype: int64


In [17]:
#Melihat jumlah duplikasi invoice dalam satu bulan (>1 = duplikat)
print(invoice_counts.value_counts())

1    185877
2     44244
3      7209
4       878
5        88
6         7
7         2
Name: count, dtype: int64


In [18]:
df_ipl_sorted = df_ipl.sort_values(
    by=['payment_status', 'invoice_id'],
    key=lambda col: col.map({'Paid': 2, 'Overdue': 1, 'Unpaid': 0}) if col.name == 'payment_status' else col
)
df_ipl_clean = df_ipl_sorted.drop_duplicates(subset=['unit_id', 'billing_month'], keep='last')

print(f"Before: {len(df_ipl):,} rows -> After: {len(df_ipl_clean):,} rows")

Before: 300,000 rows -> After: 238,305 rows


## **4. Missing Values di Contact Number**

In [19]:
#MCAR karena banyak faktor yang tidak terhubung (Lupa input no telp, Privacy, dll)

In [20]:
missing = df_units["contact_number"].isna().sum().sum()
total_row = len(df_units["contact_number"])
missing_pct = missing / total_row * 100
print(missing_pct)

20.0


In [21]:
df_units['contact_number_missing'] = df_units['contact_number'].isnull()

print(f"{df_units['contact_number_missing'].sum():,} units missing a phone number "
      f"({df_units['contact_number_missing'].mean():.1%})")

5,000 units missing a phone number (20.0%)


## **5. Handover Date**

In [22]:
df_ipl_clean["payment_date"] = pd.to_datetime(df_ipl_clean["payment_date"])
df_units["handover_date"] = pd.to_datetime(df_units["handover_date"])

df_ipl_clean = df_ipl_clean.drop(columns=["handover_date"], errors="ignore")
df_ipl_clean = df_ipl_clean.merge(df_units[["unit_id", "handover_date"]], on="unit_id", how="left")

# Cek apakah ada unit_id di billing yang tidak ada pasangan di df_units
unmatched = df_ipl_clean["handover_date"].isnull().sum()
print(f"Baris tanpa handover_date setelah merge (harus 0): {unmatched}")

df_ipl_clean["is_logical_error"] = (
    (df_ipl_clean["payment_status"] == "Paid") &
    (df_ipl_clean["payment_date"] < df_ipl_clean["handover_date"])
)

n_error = df_ipl_clean["is_logical_error"].sum()
n_paid = (df_ipl_clean["payment_status"] == "Paid").sum()
print(f"Invoices Paid before handover: {n_error:,} out of {n_paid:,} Paid invoices ({n_error/n_paid:.2%})")

error_value = df_ipl_clean.loc[df_ipl_clean["is_logical_error"], "total_amount"].sum()
n_units_affected = df_ipl_clean.loc[df_ipl_clean["is_logical_error"], "unit_id"].nunique()
print(f"Units affected: {n_units_affected:,}")
print(f"Total value of these anomalous transactions: Rp {error_value:,.0f}")

print(df_ipl_clean.loc[df_ipl_clean["is_logical_error"],
                  ["unit_id", "billing_month", "payment_date", "handover_date", "total_amount"]].head(10))

Baris tanpa handover_date setelah merge (harus 0): 0
Invoices Paid before handover: 6,210 out of 176,980 Paid invoices (3.51%)
Units affected: 5,510
Total value of these anomalous transactions: Rp 8,266,960,000
         unit_id billing_month payment_date handover_date  total_amount
61358  UNT-20834       2020-05   2020-05-02    2021-06-06    76492500.0
61376  UNT-23534       2019-01   2019-01-28    2020-02-18      627500.0
61385  UNT-20049       2020-04   2020-04-16    2021-05-30      860000.0
61389  UNT-21228       2021-04   2021-04-04    2022-09-05      425000.0
61393  UNT-01322       2019-02   2019-02-23    2020-12-12     1052500.0
61462  UNT-10371       2020-10   2020-10-22    2022-01-29      642500.0
61482  UNT-11286       2019-11   2019-11-06    2021-10-13     1687500.0
61505  UNT-19742       2018-08   2018-08-31    2020-01-08      627500.0
61512  UNT-14833       2020-01   2020-01-20    2021-03-22      545000.0
61573  UNT-15591       2020-04   2020-04-03    2021-07-29     1037500

# **Analysis**

# **Q1 Township atau Cluster yang mempunyai rasio IPL tertinggi**

### **Merge cluster dan township info untuk billing data**

In [28]:
df_master = (df_ipl_clean
             .merge(df_units[['unit_id', 'cluster_id', 'owner_name', 'is_vacant', 'contact_number_missing']],
                    on='unit_id', how='left')
             .merge(df_clusters, on='cluster_id', how='left'))

print(df_master.shape)
df_master.head(3)

(238305, 22)


,invoice_id,unit_id,billing_month,water_usage_m3,ipl_base_fee,total_amount,payment_status,payment_method,payment_date,payment_method_clean,...,handover_date,is_logical_error,is_overdue,is_arrears,cluster_id,owner_name,is_vacant,contact_number_missing,township_name,cluster_category
0,INV-IPL-00000002,UNT-09718,2023-10,26.0,500000,695000.0,Unpaid,NaN,NaT,NaN,...,2021-06-27,False,True,True,CLS-SML-029,Pemilik Properti 9717,False,False,BSD City,Premium Residential
1,INV-IPL-00000064,UNT-10952,2024-12,35.0,500000,762500.0,Unpaid,NaN,NaT,NaN,...,2020-02-21,False,True,True,CLS-SML-012,Pemilik Properti 10951,False,False,Kota Wisata,Commercial Ruko
2,INV-IPL-00000067,UNT-19773,2023-01,0.0,1500000,1500000.0,Unpaid,NaN,NaT,NaN,...,2022-05-02,False,True,True,CLS-SML-011,Pemilik Properti 19772,True,True,BSD City,Premium Residential


### Menghitung township yang masih ada tunggakan

In [46]:
df_master["is_arrear"] = df_master["payment_status"].isin(["Unpaid", "Overdue"])
township_overdue = df_master.groupby("township_name")["is_arrear"].mean().sort_values(ascending=False)
print("Township yang masih belum bayar atau overdue: \n")
print(township_overdue.apply(lambda x: f'{x:.2%}'))

Township yang masih belum bayar atau overdue: 

township_name
Kota Wisata     25.96%
NavaPark        25.77%
Grand Wisata    25.74%
Deltamas        25.70%
BSD City        25.57%
Name: is_arrear, dtype: str


### Menghitung persentase rumah yang vacant per township

In [49]:
is_vacant = df_master[df_master["is_arrear"]]
vacant_contribution = is_vacant.groupby("township_name")["is_vacant"].mean().sort_values(ascending=False)
print("Persentase Township yang memiliki rumah yang vacant.")
print(vacant_contribution.apply(lambda x: f'{x:.2%}'))

Persentase Township yang memiliki rumah yang vacant.
township_name
Deltamas        52.83%
Grand Wisata    52.19%
NavaPark        52.15%
BSD City        51.73%
Kota Wisata     51.11%
Name: is_vacant, dtype: str


### Menghitung Cluster dengan tunggakan tertinggi

In [ ]:
cluster_arrear = df_master.groupby(['township_name', 'cluster_id'])['is_arrear'].mean().sort_values(ascending=False)
print('Cluster dengan rasio tunggakan tertinggi:')
print(cluster_arrear.apply(lambda x: f'{x:.2%}'))

Top 5 cluster dengan rasio tunggakan tertinggi:
township_name  cluster_id 
Kota Wisata    CLS-SML-067    29.23%
               CLS-SML-053    29.22%
BSD City       CLS-SML-069    28.52%
Kota Wisata    CLS-SML-097    28.26%
               CLS-SML-048    28.21%
                               ...  
               CLS-SML-050    23.70%
               CLS-SML-072    23.62%
BSD City       CLS-SML-080    23.53%
Kota Wisata    CLS-SML-012    23.52%
Deltamas       CLS-SML-058    23.37%
Name: is_arrear, Length: 100, dtype: str


## Analisa IPL per Township
#### Untuk data persebaran tunggakan dan vacancy tersebar secara rata antara Township (Perbedaan persentase sangat minim). Untuk persentase rumah yang kosong (vacant) tersebar rata di angka 50%. Oleh karena ini, risiko tunggakan per township setara dengan persentase vacancy. Ini mendukung hipotesis bahwa rumah yang kosong adalah faktor utama dan bukan berbasis lokasi Township/Cluster.

# **Q2 Menghitung estimasi potensi pendapatan yang tertunda akibat tunggakan lebih dari 6 bulan berturut-turut.**

In [66]:
arrear_count_per_unit = df_master.groupby("unit_id")["is_arrear"].sum()
chronic_units = arrear_count_per_unit[arrear_count_per_unit >= 6].index

print(f"Jumlah unit kronis menunggak (>= 6 invoice Unpaid/Overdue): {len(chronic_units):,} dari {df_master['unit_id'].nunique():,} unit")


Jumlah unit kronis menunggak (>= 6 invoice Unpaid/Overdue): 2,592 dari 25,000 unit


### Menghitung Total Pendapatan yang hilang

In [70]:
lost_revenue = df_master[df_master["unit_id"].isin(chronic_units) & df_master["is_arrear"]]["total_amount"].sum()
print(f"Total potensi pendapatan hilang atau tertunda dari unit-unit ini: Rp {lost_revenue:,.0f}")

Total potensi pendapatan hilang atau tertunda dari unit-unit ini: Rp 22,024,950,000


### Persentase unit vacant


In [71]:
pct_chronic_vacant = df_units.set_index('unit_id').loc[chronic_units, 'is_vacant'].mean()
print(f"{pct_chronic_vacant:.1%} berstatus Vacant")

96.9% berstatus Vacant


In [ ]:
print(f"Jumlah unit kronis : {len(chronic_units):,}")
print(f"Total outstanding : Rp {potential_revenue_lost:,.0f}")
print(f"Rata-rata tunggakan per unit : Rp {potential_revenue_lost/len(chronic_units):,.0f}")

Jumlah unit kronis : 2,592
Total outstanding : Rp 22,024,950,000
Rata-rata tunggakan per unit : Rp 8,497,280


# **Q3. Standarisasi Payment Method**

### Sudah dijalankan ditahap data cleaning

In [76]:
print("Before:")
print(df_ipl["payment_method"].value_counts(dropna=False))
print("\nAfter:")
print(df_ipl["payment_method_clean"].value_counts(dropna=False))

Before:
payment_method
NaN                    90278
toped                  23449
KAS                    23438
Cash                   23370
cc                     23302
Credit Card            23267
bca va                 23242
Tokopedia              23238
BCA Virtual Account    23216
VA BCA                 23200
Name: count, dtype: int64

After:
payment_method_clean
NaN                    90278
BCA Virtual Account    69658
Cash                   46808
Tokopedia              46687
Credit Card            46569
Name: count, dtype: int64


# **Q4. Persentasi No Telp yang kosong bagi pemilik rumah**

### Kita tidak mengubah atau menghapus no telp sebagai kosong karena no telp adalah data yang kategorikal (Qualitatif)

In [77]:
df_units['contact_number_missing'] = df_units['contact_number'].isnull()

print(f"{df_units['contact_number_missing'].sum():,} units missing a phone number "
      f"({df_units['contact_number_missing'].mean():.1%})")

5,000 units missing a phone number (20.0%)


# **Q5. Anomali tagihan IPL berstatus "Paid" dengan tanggal pembayaran sebelum diserahterimakan (Handover)**

In [81]:
print(f"Invoices Paid before handover: {n_error:,} out of {n_paid:,} Paid invoices ({n_error/n_paid:.2%})")

print(f"\nUnits affected: {n_units_affected:,}")
print(f"\nTotal value of these anomalous transactions: Rp {error_value:,.0f}")

print(df_ipl_clean.loc[df_ipl_clean["is_logical_error"],
                  ["unit_id", "billing_month", "payment_date", "handover_date", "total_amount"]].head(10))

Invoices Paid before handover: 6,210 out of 176,980 Paid invoices (3.51%)

Units affected: 5,510

Total value of these anomalous transactions: Rp 8,266,960,000
         unit_id billing_month payment_date handover_date  total_amount
61358  UNT-20834       2020-05   2020-05-02    2021-06-06    76492500.0
61376  UNT-23534       2019-01   2019-01-28    2020-02-18      627500.0
61385  UNT-20049       2020-04   2020-04-16    2021-05-30      860000.0
61389  UNT-21228       2021-04   2021-04-04    2022-09-05      425000.0
61393  UNT-01322       2019-02   2019-02-23    2020-12-12     1052500.0
61462  UNT-10371       2020-10   2020-10-22    2022-01-29      642500.0
61482  UNT-11286       2019-11   2019-11-06    2021-10-13     1687500.0
61505  UNT-19742       2018-08   2018-08-31    2020-01-08      627500.0
61512  UNT-14833       2020-01   2020-01-20    2021-03-22      545000.0
61573  UNT-15591       2020-04   2020-04-03    2021-07-29     1037500.0


# **Q6. Ditemukan 3000 transaksi dengan pemakaian air yang tidak wajar (Dikarenakan input error atau human error)**

In [82]:
print(df_ipl['water_usage_flag'].value_counts())
print(f"\nTotal baris anomali: {(df_ipl['water_usage_flag'] != 'Normal').sum()} dari {len(df_ipl)} ({(df_ipl['water_usage_flag'] != 'Normal').mean():.2%})")

water_usage_flag
Normal                           297000
Negative (Input Error)             1519
Extreme Outlier (Human Error)      1481
Name: count, dtype: int64

Total baris anomali: 3000 dari 300000 (1.00%)
